# SAIGE — corrected ablation on free Colab

Runs `ablation_v3.py --eval-set legacy` on a **free T4**. Produces corrected
logprob-margin numbers for `base / run3 / run4 / run5` on the 11 held-out records
of the original dataset.

**What this fixes vs. the published v1/v2 tables**

1. **Completion templating.** v2 built each scored completion with
   `apply_chat_template()` on an assistant-only message list. On Qwen2.5 that
   re-emits the template's default system turn *plus* a second assistant header,
   so every response was scored after a malformed mid-conversation system turn,
   padded with 24 junk tokens. v3 scores the assistant content plus `<|im_end|>`,
   which is what the model actually emits.
2. **Statistic.** v2 reported the mean of raw per-record margins, which span
   -150 to +130 and are driven by length and style. v3 reports the **paired**
   per-record delta vs. base with an exact sign test and a bootstrap CI.
3. **Prompt confound.** The `rs` condition is now byte-identical to the prompt
   the chosen responses were generated under (they differed by a trailing
   newline through v4).

**What this cannot tell you.** These 11 records sit at mean judge delta 0.42,
while run5 was trained on delta >= 3 pairs. This eval is a different distribution
from run5's training data, so it *cannot* decide whether run5 memorized or
generalized. That needs the matched held-out set, which requires the 72B
generation run. Read these numbers as a corrected baseline, not a verdict.

**Runtime:** ~15 min. **Cost:** free.


## 1. Confirm a GPU is attached

Free Colab sometimes hands you a CPU-only runtime. This cell stops early if so.

In [ ]:
import subprocess, sys

try:
    out = subprocess.check_output(["nvidia-smi"], text=True)
    print(out)
except Exception:
    raise SystemExit(
        "No GPU attached.\n"
        "Runtime > Change runtime type > Hardware accelerator: T4 GPU, then rerun.\n"
        "Free-tier GPUs are not guaranteed; if T4 is unavailable, try again later."
    )

import torch
assert torch.cuda.is_available(), "torch cannot see the GPU"
name = torch.cuda.get_device_name(0)
vram = torch.cuda.get_device_properties(0).total_memory / 1e9
cc = torch.cuda.get_device_capability(0)
print(f"{name} | {vram:.1f} GB | compute capability {cc[0]}.{cc[1]}")
if cc[0] < 8:
    print("Turing-class card: no bf16. The script already uses fp16 + NF4 for this.")
if vram < 14:
    print("WARNING: under 14 GB. The 3B in 4-bit should still fit, but watch for OOM.")

## 2. Install dependencies

Colab's preinstalled `torch` is kept as-is; only the pieces the script needs are added.

In [ ]:
%pip install -q -U "transformers>=4.45" "datasets>=2.20" "peft>=0.11" "bitsandbytes>=0.43" "accelerate>=0.30" huggingface_hub

import importlib, transformers, datasets, peft, bitsandbytes, torch
for m in (torch, transformers, datasets, peft, bitsandbytes):
    print(f"{m.__name__:16s} {m.__version__}")

## 3. Get `ablation_v3.py`

Tries the Hub first. If you haven't uploaded it yet, it falls back to Colab's
file picker — choose the `ablation_v3.py` you were sent.

In [ ]:
import os
from pathlib import Path

SCRIPT = Path("ablation_v3.py")
REPO = "M1ztyk/saige-2x2-ablation-results"

if not SCRIPT.exists():
    try:
        from huggingface_hub import hf_hub_download
        p = hf_hub_download(repo_id=REPO, filename="ablation_v3.py", repo_type="dataset")
        SCRIPT.write_bytes(Path(p).read_bytes())
        print(f"downloaded from {REPO}")
    except Exception as e:
        print(f"not on the Hub yet ({type(e).__name__}) - upload it instead")
        from google.colab import files
        up = files.upload()
        assert "ablation_v3.py" in up, "please select ablation_v3.py"
        SCRIPT.write_bytes(up["ablation_v3.py"])

assert SCRIPT.exists() and SCRIPT.stat().st_size > 10_000, "script missing or truncated"
print(f"ablation_v3.py ready ({SCRIPT.stat().st_size} bytes)")

## 4. Dry run first

CPU-only, no model load, under a minute. It validates the data split, the
templating, the BPE prefix property, and the statistics — and asserts that the
v2 templating bug is *not* reproduced.

**If this fails, stop.** Nothing below will work and you'd waste GPU time.

In [ ]:
!ABLATION_DRY_RUN=1 python ablation_v3.py --eval-set legacy

## 5. Score it

Four states x three prompt conditions x 11 records. The model is loaded fresh per
state so no adapter can leak into another; that costs a few minutes of reloading
and buys clean separation.

`--no-push` keeps everything local. To push instead, run
`huggingface_hub.login()` with a **write** token and drop the flag.

In [ ]:
!python ablation_v3.py --eval-set legacy --no-push

## 6. Read the result

The headline is the **paired delta vs. base**, not the raw margin. Raw margins
are dominated by response length; preference accuracy is pinned by the base
model's prior and barely moves between adapters, which is exactly why v2's table
was uninformative.

`n_positive / n_nonzero` with `sign_test_p` is the number that matters: it asks
whether an adapter moved records *consistently in one direction*, which a mean
over wildly-scaled margins cannot tell you.

In [ ]:
import json
from pathlib import Path

res = json.loads(Path("ablation_results_v3.json").read_text())
ev = res["eval_sets"]["legacy"]

print("=" * 78)
print("ABSOLUTE mean margin (nats) - context only, do NOT read as the result")
print("=" * 78)
print(f"{'state':6s}" + "".join(f"{c:>12s}" for c in res["conditions"]))
for state in res["adapters"]:
    row = f"{state:6s}"
    for c in res["conditions"]:
        m = ev["absolute"][state][c]["mean_margin"]
        row += f"{m:>12.2f}"
    print(row)

print()
print("=" * 78)
print("PAIRED delta vs base - this is the result")
print("=" * 78)
hdr = f"{'state':6s} {'cond':8s} {'mean d':>9s} {'pos/n':>8s} {'sign p':>9s} {'CI95':>20s}"
print(hdr)
print("-" * len(hdr))
for state, conds in ev["paired_vs_base"].items():
    for c, st in conds.items():
        lo, hi = st["bootstrap_ci95"]
        sig = "  *" if (st["sign_test_p"] or 1) < 0.05 else ""
        print(f"{state:6s} {c:8s} {st['mean_delta']:>+9.3f} "
              f"{st['n_positive']:>3d}/{st['n_nonzero']:<4d} "
              f"{st['sign_test_p']:>9.4f} "
              f"[{lo:>+7.2f},{hi:>+7.2f}]{sig}")

print()
print("rs minus generic gap (prompt sensitivity):")
for s, g in ev["rs_minus_generic_gap"].items():
    print(f"  {s:6s} {g:+.2f}")
print()
print("NOTE:", res["condition_notes"]["none"])
print()
print("* = sign test p < 0.05")

## 7. Save the JSON

In [ ]:
from google.colab import files
files.download("ablation_results_v3.json")

## What this does and does not settle

**Settles.** Whether run3's effect survives correct scoring. Under the old
(buggy-absolute but delta-preserving) numbers, run3 moved all 11 records positive
under both `rs` and `generic`, which is p ~ 0.001 by sign test. If that holds
here, it is a real and consistent effect — though its uniformity across records
*and* prompt conditions is as consistent with a generic cause (opener style,
length) as with Right Speech. A logprob margin cannot separate those.

**Does not settle.** Whether run5 memorized or generalized. These 11 records are
low-contrast (mean judge delta 0.42) and run5 trained on delta >= 3 pairs, so a
flat result here is expected either way.

**Still blocked, and why.** Both remaining steps need the 72B judge
(`Qwen2.5-72B-Instruct-AWQ`, 41.6 GB of weights). That exceeds a free T4 by 2.6x
and also exceeds the 40 GB A100 on Colab Pro+, so paying for Colab does not
unlock it:

- `generate_rejects_v5.py` - builds the matched held-out split (72B)
- `train_dpo_v6.py` - needs that split to exist
- `eval_generation.py` - generation is T4-friendly; judging is not

The generation eval is the one that would distinguish "learned Right Speech" from
"learned nine persona signatures", because it scores what the model actually
emits rather than its logprobs on someone else's fixed text.
